Здесь представлен пайплайн обработки данных для исследования семантики мест в корпусе устных свидетельств.

1. Парсинг первичной тематической разметки с платформы «Сфира» АНО Научно-гуманитарный центр Сэфер.

2. Фильтрация фрагментов интервью, релевантных для анализа семантики мест.

3. Подготовка текстовых файлов для загрузки в QualCoder для вторичного кодирования, т.к. этот инструмент не принимает другой формат файлов.

4. Восстановление fragment_id после ручного семантического кодирования в QualCoder.

5. Соединение размеченных данных с реестром мест (который велся параллельно с вторичным семантическим кодированием) и добавление текста фрагментов для наглядности.

6. Выделение незакодированных фрагментов после первого применения инструмента для полуавтоматической разметки.

7. Объединение пилотной семантической разметки с экспортами разметки, выполненными после полуавтоматического семантического кодирования.

8. Нормализация финального датасета и группировка семантических кодов по местам с использование реестра мест.

**Скрипт 1: Парсинг размеченных интервью с платформы «Сфира».**

Назначение: загружает страницы редактирования интервью по их ID, извлекает разбивку по минутам, тексты минутных фрагментов и назначенные им теги (ключевые слова, персоналии, географию). Сохраняет результат в parsed_interviews.csv.

Входные данные: список ID интервью, cookie-строка из браузера (необходима для доступа к страницам).

Выходные данные: parsed_interviews.csv – таблица с колонками: interview_id, interview_code, minute, text, keywords, personalities, geography, fragment_id.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

# Вставляем свою cookie-строку из браузера после авторизации на sfira.org
COOKIE_STRING = "_ga=GA1.1.884313440.1764920522; _identity=d1de8d078fefe1ce9a966f1bdf75d8ab8c8cc1c2b47080c4449d3fe3bb17ad97a%3A2%3A%7Bi%3A0%3Bs%3A9%3A%22_identity%22%3Bi%3A1%3Bs%3A47%3A%22%5B90%2C%22fDnERVyteCFTPL8mxSpEbr79NG_g3S_z%22%2C2592000%5D%22%3B%7D; _ga_2Y4HY49SE7=GS2.1.s1767633003$o5$g0$t1767633003$j60$l0$h0; PHPSESSID=agqup7eokqkljggib0800fvgfj; _csrf=60f9acc681f062e328c1224bc3397cc5fe5ff15243bc49eacda30f352500bfc4a%3A2%3A%7Bi%3A0%3Bs%3A5%3A%22_csrf%22%3Bi%3A1%3Bs%3A32%3A%22iGJbDM0OV4TNuzdvaMd8fPWeTqQQEx0q%22%3B%7D"
interview_ids = [1087, 1110, 1107, 1126, 1104, 1116, 1123, 1105, 1102, 1119, 1120, 1106, 1115, 1088, 1108, 1096, 1117, 1090, 1124, 1109, 1136, 1122, 1101, 1111, 1114, 1137, 1112, 1127, 1131, 1118]  # ваши ID

cookies = {}
for item in COOKIE_STRING.split('; '):
    if '=' in item:
        key, val = item.split('=', 1)
        cookies[key] = val

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

def fetch_interview_page(interview_id):
    url = f'https://sfira.org/backend/web/interview/update?id={interview_id}'
    try:
        response = requests.get(url, headers=HEADERS, cookies=cookies, timeout=15)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Ошибка загрузки интервью {interview_id}: {e}")
        return None

def parse_interview(html, interview_id):
    soup = BeautifulSoup(html, 'html.parser')
    title_input = soup.find('input', {'id': 'interview-title'}) or soup.find('input', {'name': 'Interview[title]'})
    interview_code = title_input.get('value').strip() if title_input else f"ID_{interview_id}"
    rows = soup.select('tr.multiple-input-list__item')
    if not rows:
        print(f"Интервью {interview_id}: не найдены строки с текстом.")
        return []
    data = []
    for row in rows:
        minute_cell = row.find('td', class_='list-cell__minute')
        minute = minute_cell.find('input').get('value') if minute_cell else None
        text_cell = row.find('td', class_='list-cell__text')
        textarea = text_cell.find('textarea') if text_cell else None
        text = textarea.get_text(strip=True) if textarea else None
        keywords = extract_tags(row, 'keywordsTags')
        personalities = extract_tags(row, 'personalitiesTags')
        geography = extract_tags(row, 'geographyTags')
        data.append({
            'minute': minute,
            'text': text,
            'keywords': ', '.join(keywords),
            'personalities': ', '.join(personalities),
            'geography': ', '.join(geography),
            'interview_code': interview_code,
            'interview_id': interview_id
        })
    return data

def extract_tags(row, tag_type):
    cell = row.find('td', class_=f'list-cell__{tag_type}')
    if not cell:
        return []
    select_tag = cell.find('select', {'name': re.compile(rf'Interview\[texts\]\[\d+\]\[{tag_type}\]')})
    if not select_tag:
        return []
    selected = select_tag.find_all('option', selected=True)
    tags = []
    for opt in selected:
        raw = opt.get_text()
        clean = raw.replace('\\u00a0', ' ').strip()
        clean = re.sub(r'^[\\s=–—-]+', '', clean).strip()
        clean = re.sub(r'^[.\\s]+', '', clean).strip()
        if clean:
            tags.append(clean)
    return tags

all_records = []
for idx, iid in enumerate(interview_ids, 1):
    print(f"[{idx}/{len(interview_ids)}] Обрабатываю интервью ID {iid}...")
    html = fetch_interview_page(iid)
    if html:
        records = parse_interview(html, iid)
        if records:
            all_records.extend(records)
            print(f"    -> найдено {len(records)} фрагментов")
        else:
            print(f"    -> нет текстовых фрагментов")
    else:
        print(f"    -> пропущено (ошибка загрузки)")
    time.sleep(1)

if all_records:
    df = pd.DataFrame(all_records)
    df = df[['interview_id', 'interview_code', 'minute', 'text', 'keywords', 'personalities', 'geography']]
    df['fragment_id'] = df['interview_id'].astype(str) + '_' + df['minute'].astype(str)
    df.to_csv('parsed_interviews.csv', index=False, encoding='utf-8')
    print(f"\n✅ Готово! Сохранено {len(df)} фрагментов из {len(interview_ids)} интервью.")
    print("Файл: parsed_interviews.csv")
else:
    print("❌ Не удалось собрать ни одного фрагмента.")

[1/30] Обрабатываю интервью ID 1087...
    -> найдено 87 фрагментов
[2/30] Обрабатываю интервью ID 1110...
    -> найдено 98 фрагментов
[3/30] Обрабатываю интервью ID 1107...
    -> найдено 41 фрагментов
[4/30] Обрабатываю интервью ID 1126...
    -> найдено 57 фрагментов
[5/30] Обрабатываю интервью ID 1104...
    -> найдено 103 фрагментов
[6/30] Обрабатываю интервью ID 1116...
    -> найдено 137 фрагментов
[7/30] Обрабатываю интервью ID 1123...
    -> найдено 136 фрагментов
[8/30] Обрабатываю интервью ID 1105...
    -> найдено 144 фрагментов
[9/30] Обрабатываю интервью ID 1102...
    -> найдено 88 фрагментов
[10/30] Обрабатываю интервью ID 1119...
    -> найдено 38 фрагментов
[11/30] Обрабатываю интервью ID 1120...
    -> найдено 80 фрагментов
[12/30] Обрабатываю интервью ID 1106...
    -> найдено 72 фрагментов
[13/30] Обрабатываю интервью ID 1115...
    -> найдено 65 фрагментов
[14/30] Обрабатываю интервью ID 1088...
    -> найдено 52 фрагментов
[15/30] Обрабатываю интервью ID 1108...

**Скрипт 2: Фильтрация фрагментов для семантического кодирования**

Назначение: из всего массива данных отбираются только те фрагменты, которые содержат хотя бы один «пространственно-релевантный» тег из предопределённого списка. Результат сохраняется в filtered_grouped_for_coding.csv.

Входные данные: parsed_interviews.csv.

Выходные данные: filtered_grouped_for_coding.csv – таблица с колонками: fragment_id, interview_id, interview_code, minute, text, all_tags (объединённые релевантные теги).

In [ ]:
import pandas as pd

df = pd.read_csv('/content/parsed_interviews.csv')

# Множество тегов, релевантных для анализа семантики мест
place_tags = {
    'Еврейское пространство', 'Локальный текст', 'Городской текст', 'Культурная жизнь',
    'Еврейский театр', 'Репрезентация еврейской истории в музее', 'Этно-конфессиональные фестивали',
    'Топонимика', 'Символика', 'Сенсорные образы', 'Туризм', 'Дом и быт', 'Кошерная посуда',
    'Мезуза', 'Перестройка дома', 'Убранство дома', 'Устройство дома', 'Местечко', 'Синагога',
    'Региональная идентичность', 'Численность евреев', 'Пища', 'Кошерная/некошерная еда',
    'Повседневная еда', 'Праздничная еда', 'Смешанные браки', 'Характер', 'Добрососедство',
    'Межэтнические конфликты', 'Этнические стереотипы', 'Кладбище', 'Могила', 'Погребальный обряд',
    'Поминальные дни', 'Похороны', 'Йом Кипур', 'Песах', 'Пурим', 'Рош-ха-Шана', 'Суббота',
    'Суккот', 'Ханука', 'Шавуот', 'Раввин', 'Молитва', 'Молитвенные принадлежности',
    'Ритуальное омовение', 'Рабанит', 'Вторая мировая война', 'Фронт', 'Тыл', 'Эвакуация',
    'Холокост', 'Мемориализация', 'Образование', 'Еврейские школы', 'Религиозное образование',
    'Семейная история', 'Происхождение фамилии', 'Профессия', 'Переезд', 'Советская еврейская история',
    'Биробиджанский проект', 'Первые переселенцы', 'Землячества', 'Предприятия региона',
    'Репрессии', 'Еврейские колхозы', 'Современная еврейская жизнь', 'Еврейские организации',
    'Общинная жизнь', 'Телевидение и пресса', 'Эмиграция', 'Репатриация в Израиль (алия)',
    'Легенды', 'Предания', 'Слухи и толки', 'Неевреи в синагоге', 'Неевреи на еврейском кладбище',
    'Еврейская речь', 'Идиш', 'Иврит', 'Языковая политика'
}

def extract_relevant_tags(row):
    tags = []
    if pd.notna(row['geography']):
        tags.extend([t.strip() for t in row['geography'].split(',') if t.strip()])
    if pd.notna(row['keywords']):
        tags.extend([t.strip() for t in row['keywords'].split(',') if t.strip()])
    if pd.notna(row['personalities']):
        tags.extend([t.strip() for t in row['personalities'].split(',') if t.strip()])
    return [tag for tag in tags if tag in place_tags]

df['relevant_tags'] = df.apply(extract_relevant_tags, axis=1)
df_filtered = df[df['relevant_tags'].apply(len) > 0].copy()
df_filtered['all_tags'] = df_filtered['relevant_tags'].apply(lambda x: '; '.join(x))

result = df_filtered[['fragment_id', 'interview_id', 'interview_code', 'minute', 'text', 'all_tags']]
result.to_csv('filtered_grouped_for_coding.csv', index=False, encoding='utf-8')
print(f'Сохранено фрагментов, подходящих для семантического кодирования: {len(result)}')

Сохранено фрагментов, подходящих для семантического кодирования: 2205


**Скрипт 3: Подготовка текстовых файлов для QualCoder**

Назначение: преобразует отфильтрованную таблицу в набор текстовых файлов (по одному на интервью), которые можно загрузить в программу для качественного анализа QualCoder. Каждый файл содержит фрагменты с указанием fragment_id, минуты и исходных тегов.

Входные данные: filtered_grouped_for_coding.csv.

Выходные данные: папка taguette_input с файлами вида EAO_25_XX_YYY.txt.

In [ ]:
import pandas as pd
import os

df = pd.read_csv('/content/filtered_grouped_for_coding.csv')
output_dir = 'taguette_input'
os.makedirs(output_dir, exist_ok=True)

for code, group in df.groupby('interview_code'):
    filename = f"{code}.txt".replace('/', '_').replace('\\', '_')
    with open(os.path.join(output_dir, filename), 'w', encoding='utf-8') as f:
        for _, row in group.iterrows():
            f.write(f"=== fragment_id: {row['fragment_id']} ===\n")
            f.write(f"minute: {row['minute']}\n")
            f.write(f"первичные_теги: {row['all_tags']}\n")
            f.write("текст:\n")
            f.write(row['text'].replace('\n', ' ').strip() + '\n\n')
    print(f"Создан файл: {output_dir}/{filename}")

Создан файл: taguette_input/EAO_25_11_Bb.txt
Создан файл: taguette_input/EAO_25_12_Bb.txt
Создан файл: taguette_input/EAO_25_19_Am.txt
Создан файл: taguette_input/EAO_25_22_Bb.txt
Создан файл: taguette_input/EAO_25_23_Bb.txt
Создан файл: taguette_input/EAO_25_29_Bb.txt
Создан файл: taguette_input/EAO_25_35_Bb.txt
Создан файл: taguette_input/EAO_25_40_Vald.txt
Создан файл: taguette_input/EAO_25_41_Vald.txt
Создан файл: taguette_input/EAO_25_42_Vald.txt
Создан файл: taguette_input/EAO_25_44_Vald.txt


**Скрипт 4: Восстановление fragment_id из разметки QualCoder**

Назначение: после того как мы вручную разметили фрагменты в QualCoder и экспортировали разметку в CSV (файл Разметка_пилот.csv), этот скрипт сопоставляет каждую размеченную строку с исходным fragment_id по совпадению текста и имени файла. Группирует коды и заметки по fragment_id и сохраняет результат в pilot_coded_grouped.csv.

Входные данные:



*   экспорт из QualCoder (например, Разметка_пилот.csv),
*   filtered_grouped_for_coding.csv.





Выходные данные: pilot_coded_grouped.csv – таблица с колонками: fragment_id, minute, File, codes, memos.

In [ ]:
import pandas as pd

export = pd.read_csv('/content/export_qc.csv')
source = pd.read_csv('/content/filtered_grouped_for_coding.csv')
source['file_name'] = source['interview_code'] + '.txt'

def find_fragment(row, source_df):
    text = row['Coded']
    file = row['File']
    matches = source_df[(source_df['file_name'] == file) &
                        (source_df['text'].str.contains(text, na=False, regex=False))]
    if len(matches) == 1:
        return matches.iloc[0]['fragment_id'], matches.iloc[0]['minute']
    elif len(matches) > 1:
        print(f"Предупреждение: для текста '{text[:50]}...' найдено {len(matches)} совпадений")
        return matches.iloc[0]['fragment_id'], matches.iloc[0]['minute']
    else:
        return None, None

result = export.apply(lambda r: pd.Series(find_fragment(r, source)), axis=1)
result.columns = ['fragment_id', 'minute']
merged = pd.concat([export, result], axis=1)
merged = merged.dropna(subset=['fragment_id'])

grouped = merged.groupby(['fragment_id', 'minute', 'File']).agg({
    'Codename': lambda x: '; '.join(x),
    'Coded_Memo': lambda x: '; '.join([str(m) for m in x if pd.notna(m)])
}).reset_index()
grouped.rename(columns={'Codename': 'codes', 'Coded_Memo': 'memos'}, inplace=True)

grouped.to_csv('pilot_coded_grouped.csv', index=False)
print(f'Обработано фрагментов: {len(grouped)}')

Обработано фрагментов: 186


In [ ]:
# Загружаем только что созданный файл
df = pd.read_csv('/content/pilot_coded_grouped.csv')

# Нормализуем fragment_id: добавляем '.0', если нет точки
df['fragment_id'] = df['fragment_id'].apply(lambda x: x if '.' in str(x) else str(x) + '.0')

# Сохраняем обратно
df.to_csv('/content/pilot_coded_grouped.csv', index=False, encoding='utf-8')

In [ ]:
# Загружаем filtered_grouped_for_coding.csv
source_text = pd.read_csv('/content/filtered_grouped_for_coding.csv')

# Нормализуем fragment_id: добавляем '.0', если нет точки
source_text['fragment_id'] = source_text['fragment_id'].apply(lambda x: x if '.' in str(x) else str(x) + '.0')

# Сохраняем обратно
source_text.to_csv('/content/filtered_grouped_for_coding.csv', index=False, encoding='utf-8')

**Скрипт 5: Соединение с реестром мест и добавление текста**

Назначение: объединяет размеченные данные (pilot_coded_grouped.csv) с реестром мест (reestr_place.csv), где каждому месту сопоставлены fragment_id. Добавляет текст фрагментов из исходного файла filtered_grouped_for_coding.csv. Результат – полная таблица для анализа семантики мест.

Входные данные:



*   reestr_place.csv
*   pilot_coded_grouped.csv
*   filtered_grouped_for_coding.csv







Выходные данные: places_with_codes_and_text.csv – финальная таблица с колонками: place_id, name_place, fragment_id, minute, File, codes, memos, text.

In [ ]:
import pandas as pd

# 1. Загружаем реестр мест
places = pd.read_csv('/content/reestr_place.csv', sep=';')
places = places.iloc[:, :3]

# 2. Загружаем размеченные данные (после восстановления fragment_id)
coded = pd.read_csv('/content/pilot_coded_grouped.csv')

# 3. Добавляем текст и первичные теги из исходного файла
source = pd.read_csv('/content/filtered_grouped_for_coding.csv')
# Берём уникальные fragment_id с текстом и тегами
source_unique = source[['fragment_id', 'text', 'all_tags']].drop_duplicates(subset=['fragment_id'])
# Присоединяем к coded
coded = coded.merge(source_unique, on='fragment_id', how='left')

# 4. Разворачиваем fragment_id в реестре мест
places['fragment_list'] = places['fragment_id'].str.split(',')
places_expanded = places.explode('fragment_list')
places_expanded['fragment_list'] = places_expanded['fragment_list'].str.strip()
places_expanded = places_expanded.drop(columns=['fragment_id'])
places_expanded = places_expanded.rename(columns={'fragment_list': 'fragment_id'})

# 5. Объединяем
merged = pd.merge(places_expanded, coded, on='fragment_id', how='left')

# 6. Сохраняем финальный результат
merged.to_csv('places_with_codes_and_text.csv', index=False, encoding='utf-8-sig')
print("✅ Файл places_with_codes_and_text.csv сохранён.")
print("Колонки:", merged.columns.tolist())

✅ Файл places_with_codes_and_text.csv сохранён.
Колонки: ['place_id', 'name_place', 'fragment_id', 'minute', 'File', 'codes', 'memos', 'text', 'all_tags']


После выполнения всех пяти скриптов мы получили файл **places_with_codes_and_text.csv**, в котором для каждого места собраны все относящиеся к нему фрагменты интервью, семантические коды и заметки.

Это позволяет нам:

*   анализировать семантическую насыщенность каждого места и составлять его портрет

*   сравнивать, как одно и то же место наделяется различными смыслами разными информантами
*   анализировать соотношение тематической и семантической разметки

*   отбирать значимые точки для цифрового туристического маршрута

Все промежуточные файлы (parsed_interviews.csv, filtered_grouped_for_coding.csv, pilot_coded_grouped.csv) также сохраняются и могут быть использованы для дополнительных проверок или других аналитических задач.

In [ ]:
semantic_place_pilot = pd.read_csv('/content/places_with_codes_and_text.csv')
semantic_place_pilot.head(10)

,place_id,name_place,fragment_id,minute,File,codes,memos,text,all_tags
0,amurzet,Амурзет,1089_0.0,0.0,EAO_25_19_Am.txt,narrative_crossing; place_settlement; relation...,"place_id: mogilev_podolsky, place_id: amurzet",[Жителя Октябрьского района $Михаила Исаакович...,Вторая мировая война; Семейная история; Амурзе...
1,amurzet,Амурзет,1089_16.0,16.0,EAO_25_19_Am.txt,narrative_crossing; place_settlement; relation...,"place_id: amurzet, place_id: bir",Исаака Лейбовича Кадемии. И наша бригада – кли...,Локальный текст; Топонимика; Семейная история;...
2,amurzet,Амурзет,1089_17.0,17.0,EAO_25_19_Am.txt,place_settlement; process_creation; relation_w...,place_id: amurzet,"И получилось, что мы - биробиджане - займем др...",Локальный текст; Семейная история; Профессия; ...
3,amurzet,Амурзет,1089_19.0,19.0,EAO_25_19_Am.txt,place_settlement; process_identity; relation_i...,place_id: amurzet,уже стало самостоятельно строительное монтажно...,Локальный текст; Советская еврейская история; ...
4,amurzet,Амурзет,1089_21.0,21.0,EAO_25_19_Am.txt,place_settlement; relation_event; relation_family,place_id: amurzet,Обратно бы случайность совпадение получилось. ...,Семейная история
5,amurzet,Амурзет,1089_25.0,25.0,EAO_25_19_Am.txt,place_settlement; temporal_past,place_id: amurzet,[Мы можем перефотографировать.] Вот не помню. ...,Локальный текст; Численность евреев; Советская...
6,amurzet,Амурзет,1089_27.0,27.0,EAO_25_19_Am.txt,place_settlement; temporal_past,place_id: amurzet,"на еврейском языке написать. А здесь, в Амурзе...",Образование; Семейная история; Амурзет
7,amurzet,Амурзет,1089_28.0,28.0,EAO_25_19_Am.txt,place_settlement; relation_residence,place_id: amurzet,"Но я думаю, что больше на русском языке. [А ва...",Семейная история; Идиш; ИЗРАИЛЬ
8,amurzet,Амурзет,1089_47.0,47.0,EAO_25_19_Am.txt,place_settlement; temporal_past,place_id: amurzet,"И у меня их много, я их наизусть учил. И тут п...",Синагога; Численность евреев; Песах; Молитва; ...
9,amurzet,Амурзет,1089_55.0,55.0,EAO_25_19_Am.txt,place_settlement; temporal_past,place_id: amurzet,у меня собирались и у меня справляли вот эти м...,Локальный текст; Советская еврейская история; ...


**Скрипт 6: выделение незакодированных фрагментов после первого применения инструмента для полуавтоматической разметки**

Назначение: из всего массива фрагментов, прошедших первичную фильтрацию, исключает те интервью, которые уже были обработаны с помощью инструмента полуавтоматической разметки semantic_coder. Это позволяет увидеть, какие фрагменты ещё предстоит разметить.

Входные данные:

* filtered_grouped_for_coding (2).csv – таблица фрагментов, релевантных для анализа семантики мест.

* Список имён файлов (interview_code) уже размеченных интервью (задан вручную в коде).

Выходные данные: uncoded_fragments.csv – таблица с теми же колонками, но содержащая только фрагменты из ещё не обработанных интервью.

In [ ]:
import pandas as pd

# Загружаем все фрагменты
df = pd.read_csv('/content/filtered_grouped_for_coding (2).csv')

# Список интервью, которые уже размечены (коды из колонки File)
coded_files = ['EAO_25_09_Bb', 'EAO_25_16_Bb', 'EAO_25_18_Bf', 'EAO_25_21_Bb']

# Оставляем только те строки, где interview_code НЕ входит в список размеченных
uncoded = df[~df['interview_code'].isin(coded_files)]

# Сохраняем результат
uncoded.to_csv('uncoded_fragments.csv', index=False, encoding='utf-8')
print(f"Осталось фрагментов: {len(uncoded)}")

Осталось фрагментов: 1963


**Скрипт 7: Объединение пилотной семантической разметки с экспортами, выполненными после полуавтоматического семантического кодирования**

Назначение: собирает в единый датасет все результаты ручного семантического кодирования. Использует несколько источников: пилотный размеченный датасет, экспорты из QualCoder (или аналогичного инструмента) в виде CSV-файлов. Для каждого фрагмента извлекает из заметок (memos) идентификаторы мест (place_id), подтягивает недостающие тексты и первичные теги из исходного файла filtered_grouped_for_coding (2).csv. Удаляет дубликаты по fragment_id.

Входные данные:

* places_with_codes_and_text (2).csv – пилотный датасет с семантическими кодами (244 фрагмента).

* semantic_results (1).csv, semantic_results (2).csv, semantic_results (3).csv – экспорты разметки из инструмента кодирования.

* filtered_grouped_for_coding (2).csv – исходная таблица фрагментов (для добавления текста и первичных тегов).

Выходные данные: all_coded.csv – объединенная таблица с колонками: fragment_id, place_id, name_place, minute, File, codes, memos, text, all_tags.

In [ ]:
import pandas as pd
import re

# ------------------------------
# 0. Загружаем filtered_grouped_for_coding.csv для дополнения отсутствующих колонок
# ------------------------------
source = pd.read_csv('/content/filtered_grouped_for_coding (2).csv')
source = source[['fragment_id', 'text', 'all_tags', 'interview_code', 'minute']]

# ------------------------------
# 1. Загрузка пилотного датасета
# ------------------------------
pilot = pd.read_csv('/content/places_with_codes_and_text (2).csv')
print(f"Пилот: {len(pilot)} фрагментов")

# ------------------------------
# 2. Функция обработки экспорта
# ------------------------------
def process_export(file_path):
    df = pd.read_csv(file_path)
    # Фильтруем строки с непустыми codes
    df = df[df['codes'].notna() & (df['codes'].str.strip() != '')]

    # Если в df нет нужных колонок, дополняем из source
    required_cols = ['text', 'all_tags', 'minute', 'interview_code']
    if not all(col in df.columns for col in required_cols):
        df = df.merge(source, on='fragment_id', how='left')
        # Проверим, что после слияния колонки появились
        for col in required_cols:
            if col not in df.columns:
                print(f"В файле {file_path} после слияния отсутствует колонка {col}")
    else:
        # Уже есть все колонки, но убедимся, что minute не целое (если нужно)
        pass

    # Извлекаем place_id из memos
    def extract_place_ids(memos_str):
        if pd.isna(memos_str):
            return ''
        matches = re.findall(r'place_id:\s*([a-zA-Z0-9_]+)', memos_str)
        return ', '.join(matches)

    df['place_id'] = df['memos'].apply(extract_place_ids)
    df['name_place'] = ''
    df['File'] = df['interview_code'] + '.txt'

    final_cols = ['fragment_id', 'place_id', 'name_place', 'minute', 'File', 'codes', 'memos', 'text', 'all_tags']
    # Проверяем наличие всех колонок
    for col in final_cols:
        if col not in df.columns:
            print(f"В файле {file_path} после обработки отсутствует колонка: {col}")
    return df[final_cols]

# Обрабатываем экспорты
export1 = process_export('/content/semantic_results (1).csv')
export2 = process_export('/content/semantic_results (2).csv')
export3 = process_export('/content/semantic_results (3).csv')
print(f"Экспорт 1: {len(export1)} фрагментов")
print(f"Экспорт 2: {len(export2)} фрагментов")
print(f"Экспорт 3: {len(export3)} фрагментов")

# ------------------------------
# 3. Объединение (порядок: пилот, экспорт1, экспорт2 – последний имеет приоритет)
# ------------------------------
all_coded = pd.concat([pilot, export1, export2, export3], ignore_index=True, sort=False)
print(f"До удаления дубликатов: {len(all_coded)}")
all_coded = all_coded.drop_duplicates(subset='fragment_id', keep='last')
print(f"После удаления дубликатов: {len(all_coded)}")

# ------------------------------
# 4. Сохранение
# ------------------------------
all_coded.to_csv('all_coded.csv', index=False, encoding='utf-8')
print("Сохранён файл all_coded.csv")

Пилот: 244 фрагментов
Экспорт 1: 75 фрагментов
Экспорт 2: 32 фрагментов
Экспорт 3: 161 фрагментов
До удаления дубликатов: 512
После удаления дубликатов: 409
Сохранён файл all_coded.csv


In [ ]:
df_final = pd.read_csv('/content/all_coded.csv')
df_final

,place_id,name_place,fragment_id,minute,File,codes,memos,text,all_tags
0,amurzet,Амурзет,1089_17.0,17.0,EAO_25_19_Am.txt,place_settlement; process_creation; relation_w...,place_id: amurzet,"И получилось, что мы - биробиджане - займем др...",Локальный текст; Семейная история; Профессия; ...
1,amurzet,Амурзет,1089_19.0,19.0,EAO_25_19_Am.txt,place_settlement; process_identity; relation_i...,place_id: amurzet,уже стало самостоятельно строительное монтажно...,Локальный текст; Советская еврейская история; ...
2,amurzet,Амурзет,1089_21.0,21.0,EAO_25_19_Am.txt,place_settlement; relation_event; relation_family,place_id: amurzet,Обратно бы случайность совпадение получилось. ...,Семейная история
3,amurzet,Амурзет,1089_25.0,25.0,EAO_25_19_Am.txt,place_settlement; temporal_past,place_id: amurzet,[Мы можем перефотографировать.] Вот не помню. ...,Локальный текст; Численность евреев; Советская...
4,amurzet,Амурзет,1089_27.0,27.0,EAO_25_19_Am.txt,place_settlement; temporal_past,place_id: amurzet,"на еврейском языке написать. А здесь, в Амурзе...",Образование; Семейная история; Амурзет
...,...,...,...,...,...,...,...,...,...
404,reka_bira,NaN,1096_75,75.0,EAO_25_42_Vald.txt,temporal_past; place_natural,place_id: reka_bira,"АС: Два раза в год топили. Вот просыпаешься, в...",Городской текст
405,vald,NaN,1096_76,76.0,EAO_25_42_Vald.txt,place_settlement; space_name; temporal_past,place_id: vald,"[ВШ: Ну, конечно.] ВИ: По реке, Головино по ре...",Топонимика
406,vald,NaN,1096_78,78.0,EAO_25_42_Vald.txt,place_settlement; space_name,place_id: vald,"[ВШ: И что они потом голосовали, как назвать?]...",Локальный текст; Топонимика
407,vald,NaN,1096_79,79.0,EAO_25_42_Vald.txt,place_settlement; symbol_urban; temporal_past,place_id: vald,ВИ: Вдоль дороги у нас сейчас деревья. Строили...,Локальный текст; Биробиджанский проект


**Скрипт 8: Нормализация финального датасета и группировка семантических кодов по местам с использование Реестра мест**

Назначение: приводит идентификаторы мест в all_coded.csv к единому стандарту на основе реестра мест (reestr_places.csv). Для каждого fragment_id из реестра берутся канонические place_id и name_place. Если фрагменту в реестре соответствует несколько мест, строка размножается (по одному месту на строку). Если фрагмента в реестре нет, сохраняется исходный place_id, извлечённый из заметок (очищенный от служебных префиксов). В результате получается денормализованная таблица, удобная для дальнейшего анализа (группировки по местам, подсчёта кодов и т.д.).

Входные данные:

* all_coded.csv.

* reestr_places.csv – реестр мест с колонками place_id, name_place, fragment_id.

Выходные данные:

all_coded_normalized.csv – финальная таблица с колонками: place_id, name_place, fragment_id, minute, File, codes, memos, text, all_tags. Каждая строка соответствует одному фрагменту и одному месту.

In [ ]:
import csv
from collections import defaultdict

# 1. Загрузка реестра мест (разделитель ;)
mapping = defaultdict(list)
with open('reestr_places.csv', 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f, delimiter=';')
    for row in reader:
        place_id = row['place_id']
        name_place = row['name_place']
        fragments = row['fragment_id'].strip()
        if not fragments:
            continue
        for frag in fragments.split(','):
            frag = frag.strip()
            if frag:
                mapping[frag].append((place_id, name_place))

print(f'Загружено {len(mapping)} уникальных fragment_id')

# 2. Чтение all_coded.csv (разделитель ,)
input_file = 'all_coded.csv'
output_file = 'all_coded_updated.csv'

with open(input_file, 'r', encoding='utf-8-sig') as infile:
    # Пробуем прочитать первую строку, чтобы определить разделитель
    first_line = infile.readline()
    infile.seek(0)  # возвращаемся в начало
    if ';' in first_line and ',' not in first_line.split(';')[0]:
        delimiter = ';'
    else:
        delimiter = ','  # по умолчанию запятая
    print(f'Используемый разделитель: "{delimiter}"')

    reader = csv.reader(infile, delimiter=delimiter, quotechar='"')
    header = next(reader)

    # Нормализуем заголовок: убираем возможные пробелы
    header = [col.strip() for col in header]
    print(f'Заголовок: {header}')

    # Определяем индексы
    try:
        idx_frag = header.index('fragment_id')
        idx_place_id = header.index('place_id')
        idx_name_place = header.index('name_place')
    except ValueError as e:
        print(f'Ошибка: в заголовке нет нужных столбцов. Доступные: {header}')
        raise

    updated_rows = [header]
    for row_num, row in enumerate(reader, start=2):
        # Если строка короче заголовка, дополним пустыми
        if len(row) < len(header):
            row += [''] * (len(header) - len(row))
        frag = row[idx_frag].strip()
        if frag in mapping:
            places = mapping[frag]
            new_place_id = '; '.join([f"place_id: {pid}" for pid, _ in places])
            new_name_place = ', '.join([name for _, name in places])
            row[idx_place_id] = new_place_id
            row[idx_name_place] = new_name_place
        updated_rows.append(row)

# 3. Сохранение
with open(output_file, 'w', encoding='utf-8-sig', newline='') as outfile:
    writer = csv.writer(outfile, delimiter=',', quotechar='"', quoting=csv.QUOTE_ALL)
    writer.writerows(updated_rows)

print(f'Готово. Обновлённый файл: {output_file}')
print(f'Всего обработано строк: {len(updated_rows)-1}')

Загружено 416 уникальных fragment_id
Используемый разделитель: ","
Заголовок: ['place_id', 'name_place', 'fragment_id', 'minute', 'File', 'codes', 'memos', 'text', 'all_tags']
Готово. Обновлённый файл: all_coded_updated.csv
Всего обработано строк: 409


In [ ]:
import csv
import re
from collections import defaultdict

# 1. Загрузка реестра мест (разделитель ;)
mapping = defaultdict(list)
with open('reestr_place_1.csv', 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f, delimiter=';')
    for row in reader:
        place_id = row['place_id']
        name_place = row['name_place']
        fragments = row['fragment_id'].strip()
        if not fragments:
            continue
        for frag in fragments.split(','):
            frag = frag.strip()
            if frag:
                mapping[frag].append((place_id, name_place))

print(f'Загружено {len(mapping)} уникальных fragment_id')

# 2. Чтение all_coded.csv
input_file = 'all_coded.csv'
output_file = 'all_coded_normalized.csv'

with open(input_file, 'r', encoding='utf-8-sig') as infile:
    first_line = infile.readline()
    infile.seek(0)
    delimiter = ','
    if ';' in first_line and ',' not in first_line.split(';')[0]:
        delimiter = ';'
    print(f'Используемый разделитель: "{delimiter}"')

    reader = csv.reader(infile, delimiter=delimiter, quotechar='"')
    header = next(reader)
    header = [col.strip() for col in header]
    print(f'Заголовок: {header}')

    idx_frag = header.index('fragment_id')
    idx_place_id = header.index('place_id')
    idx_name_place = header.index('name_place')

    new_rows = [header]

    for row_num, row in enumerate(reader, start=2):
        if len(row) < len(header):
            row += [''] * (len(header) - len(row))
        frag = row[idx_frag].strip()

        if frag in mapping and mapping[frag]:
            # Используем данные из реестра
            for pid, pname in mapping[frag]:
                new_row = row.copy()
                new_row[idx_place_id] = pid
                new_row[idx_name_place] = pname
                new_rows.append(new_row)
        else:
            old_place_id = row[idx_place_id]
            clean = re.sub(r'place_id:\s*', '', old_place_id).strip()
            if ';' in clean:
                clean = clean.split(';')[0].strip()
            row[idx_place_id] = clean
            new_rows.append(row)

# 3. Сохранение
with open(output_file, 'w', encoding='utf-8-sig', newline='') as outfile:
    writer = csv.writer(outfile, delimiter=',', quotechar='"', quoting=csv.QUOTE_ALL)
    writer.writerows(new_rows)

print(f'Готово. Файл сохранён: {output_file}')
print(f'Всего строк после нормализации: {len(new_rows)-1}')

Загружено 416 уникальных fragment_id
Используемый разделитель: ","
Заголовок: ['place_id', 'name_place', 'fragment_id', 'minute', 'File', 'codes', 'memos', 'text', 'all_tags']
Готово. Файл сохранён: all_coded_normalized.csv
Всего строк после нормализации: 480
